In [3]:
import cv2
import numpy as np
from collections import defaultdict
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score
import pandas as pd
import matplotlib.pyplot as plt
import os
from deepface import DeepFace
import datetime
import pickle
import yt_dlp

In [14]:
movie_url = pd.read_csv("movie\movie_to_photo_for_learn.csv",header=None)
movie_url.columns = ["name", "url"]
movie_url

,name,url
0,umezawa,https://www.instagram.com/reel/DYdrTa3TAU-/?ig...
1,yumiki,https://www.instagram.com/reel/DYb-r_3zkwr/?ig...
2,hayashi,https://www.instagram.com/reel/DYb7K_rTMP2/?ig...
3,kaki,https://www.instagram.com/reel/DYbP0N3TknR/?ig...
4,endou,https://www.instagram.com/reel/DYbMJh1Tcks/?ig...
5,ioki,https://www.instagram.com/reel/DYY-Jwmzj2_/?ig...
6,suzuki,https://www.instagram.com/reel/DYW9bLJTUsw/?ig...
7,ikeda,https://www.instagram.com/reel/DYZAG4TTo3b/?ig...


In [9]:
def download_instagram_video(url,name, output_path="movie"):
    ydl_opts = {
        'outtmpl': f'{output_path}/nogi_movie_{name}.mp4',
        'quiet': False,
        'no_warnings': False,
    }
    
    with yt_dlp.YoutubeDL(ydl_opts) as ydl:
        try:
            ydl.download([url])
            print("ダウンロード完了")
        except Exception as e:
            print(f"エラーが発生しました: {e}")

# 使用例
for index, row in movie_url.iterrows():
    name = row['name']
    url = row['url']
    download_instagram_video(url, name=name)

[Instagram] Extracting URL: https://www.instagram.com/reel/DYb-r_3zkwr/?igsh=MWs5cDVyd3BvYXVwMw==
[Instagram] DYb-r_3zkwr: Setting up session
[Instagram] DYb-r_3zkwr: Downloading JSON metadata


[info] DYb-r_3zkwr: Downloading 1 format(s): 8
[download] Destination: movie\nogi_movie_yumiki.mp4
[download] 100% of    7.62MiB in 00:00:00 at 10.26MiB/s  
ダウンロード完了
[Instagram] Extracting URL: https://www.instagram.com/reel/DYb7K_rTMP2/?igsh=YTBqcW14M3J2dnhj
[Instagram] DYb7K_rTMP2: Setting up session
[Instagram] DYb7K_rTMP2: Downloading JSON metadata


[info] DYb7K_rTMP2: Downloading 1 format(s): 8
[download] Destination: movie\nogi_movie_hayashi.mp4
[download] 100% of    5.81MiB in 00:00:00 at 10.92MiB/s  
ダウンロード完了
[Instagram] Extracting URL: https://www.instagram.com/reel/DYbP0N3TknR/?igsh=cG1kajB0MXdpM3Zh
[Instagram] DYbP0N3TknR: Setting up session
[Instagram] DYbP0N3TknR: Downloading JSON metadata


[info] DYbP0N3TknR: Downloading 1 format(s): 8
[download] Destination: movie\nogi_movie_kaki.mp4
[download] 100% of   11.45MiB in 00:00:00 at 14.76MiB/s    
ダウンロード完了
[Instagram] Extracting URL: https://www.instagram.com/reel/DYbMJh1Tcks/?igsh=a3Y1aWdzeGF6aTdo
[Instagram] DYbMJh1Tcks: Setting up session
[Instagram] DYbMJh1Tcks: Downloading JSON metadata


[info] DYbMJh1Tcks: Downloading 1 format(s): 8
[download] Destination: movie\nogi_movie_endou.mp4
[download] 100% of   10.44MiB in 00:00:00 at 11.24MiB/s  
ダウンロード完了
[Instagram] Extracting URL: https://www.instagram.com/reel/DYY-Jwmzj2_/?igsh=MXh4d3BocGF2bDVkZQ==
[Instagram] DYY-Jwmzj2_: Setting up session
[Instagram] DYY-Jwmzj2_: Downloading JSON metadata


[info] DYY-Jwmzj2_: Downloading 1 format(s): 8
[download] Destination: movie\nogi_movie_ioki.mp4
[download] 100% of   10.26MiB in 00:00:00 at 10.68MiB/s  
ダウンロード完了
[Instagram] Extracting URL: https://www.instagram.com/reel/DYW9bLJTUsw/?igsh=OWx5Y2dvZDhpdml5
[Instagram] DYW9bLJTUsw: Setting up session
[Instagram] DYW9bLJTUsw: Downloading JSON metadata


[info] DYW9bLJTUsw: Downloading 1 format(s): 2
[download] Destination: movie\nogi_movie_suzuki.mp4
[download] 100% of    6.47MiB in 00:00:00 at 9.21MiB/s   
ダウンロード完了
[Instagram] Extracting URL: https://www.instagram.com/reel/DYZAG4TTo3b/?igsh=M25ha2cxMW5iMDl1
[Instagram] DYZAG4TTo3b: Setting up session
[Instagram] DYZAG4TTo3b: Downloading JSON metadata


[info] DYZAG4TTo3b: Downloading 1 format(s): 2
[download] Destination: movie\nogi_movie_ikeda.mp4
[download] 100% of    6.15MiB in 00:00:00 at 12.31MiB/s  
ダウンロード完了


In [10]:
def extract_frames(video_path, output_dir):
    # ディレクトリが存在しなければ作成
    os.makedirs(output_dir, exist_ok=True)

    print(f"処理対象: {video_path}")
    print(f"ファイル存在確認: {os.path.exists(video_path)}")

    video = cv2.VideoCapture(video_path)

    # ビデオが正しく開かれたか確認
    if not video.isOpened():
        print("エラー：動画ファイルが開けません")
    else:
        fps = int(video.get(cv2.CAP_PROP_FPS))
        total_frames = int(video.get(cv2.CAP_PROP_FRAME_COUNT))
        width = int(video.get(cv2.CAP_PROP_FRAME_WIDTH))
        height = int(video.get(cv2.CAP_PROP_FRAME_HEIGHT))
        print(f"FPS: {fps}, 総フレーム数: {total_frames}, 解像度: {width}×{height}")
        
        frame_count = 0
        saved_frames = []

        while True:
            success, frame = video.read() 
            if not success:
                break  # 動画の終了

            # 1秒ごとにフレームを保存
            if frame_count % fps == 0:
                frame_time_sec = int(frame_count / fps)
                frame_file = os.path.join(output_dir, f"frame_{frame_time_sec:05d}.jpg")
                # リサイズなしで元のサイズのまま保存
                cv2.imwrite(frame_file, frame)
                saved_frames.append((frame_time_sec, frame_file))
            
            frame_count += 1

        video.release()
        print(f"✅ 処理完了！保存されたフレーム数: {len(saved_frames)}")

In [13]:
for name in movie_url['name'][1:]:
    output_dir = f"./menber_db/{name}"
    video_path = f"./movie/nogi_movie_{name}.mp4"

    extract_frames(video_path, output_dir)

処理対象: ./movie/nogi_movie_hayashi.mp4
ファイル存在確認: True
FPS: 23, 総フレーム数: 550, 解像度: 720×1280
✅ 処理完了！保存されたフレーム数: 24
処理対象: ./movie/nogi_movie_kaki.mp4
ファイル存在確認: True
FPS: 23, 総フレーム数: 1041, 解像度: 720×1280
✅ 処理完了！保存されたフレーム数: 46
処理対象: ./movie/nogi_movie_endou.mp4
ファイル存在確認: True
FPS: 23, 総フレーム数: 1054, 解像度: 720×1280
✅ 処理完了！保存されたフレーム数: 46
処理対象: ./movie/nogi_movie_ioki.mp4
ファイル存在確認: True
FPS: 23, 総フレーム数: 867, 解像度: 720×1280
✅ 処理完了！保存されたフレーム数: 38
処理対象: ./movie/nogi_movie_suzuki.mp4
ファイル存在確認: True
FPS: 23, 総フレーム数: 685, 解像度: 720×1280
✅ 処理完了！保存されたフレーム数: 30
処理対象: ./movie/nogi_movie_ikeda.mp4
ファイル存在確認: True
FPS: 23, 総フレーム数: 589, 解像度: 720×1280
✅ 処理完了！保存されたフレーム数: 26
